In [2]:
import os

# Set the environment variable
os.environ['BEN_HOME'] = '/Users/allwynsequeira/AllwynDev/ben/'

# Access the environment variable
ben_home = os.environ.get('BEN_HOME')

os.chdir(ben_home + 'src')
print(os.getcwd())



/Users/allwynsequeira/AllwynDev/ben/src


In [3]:

from urllib.parse import unquote

e_url = "https://www.bridgebase.com/myhands/hands.php?tourney=SB%3Aiit_bombay-1747543952-&username=iit_bombay"

url = unquote(e_url)
print(url)

https://www.bridgebase.com/myhands/hands.php?tourney=SB:iit_bombay-1747543952-&username=iit_bombay


In [ ]:
import time
import getpass # For securely getting password
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from bs4 import BeautifulSoup

def login_to_bridgebase(driver, username, password):
    """
    Logs into Bridge Base Online using the provided credentials.
    """
    login_url = "https://www.bridgebase.com/"
    print(f"Navigating to login page: {login_url}")
    driver.get(login_url)

    try:
        # Wait for login form elements to be present
        username_field = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "login_username"))
        )
        password_field = driver.find_element(By.ID, "login_password")
        # The login button is an input type=submit within the form id=loginform
        login_button = driver.find_element(By.CSS_SELECTOR, "form#loginform input[type='submit'][value='Login']")

        print("Entering username...")
        username_field.send_keys(username)
        print("Entering password...")
        password_field.send_keys(password)
        
        print("Clicking login button...")
        login_button.click()

        # Wait for successful login. A good indicator is the presence of a "Logout" link.
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.LINK_TEXT, "Logout"))
        )
        print("Login successful!")
        return True
    except TimeoutException:
        print("Login failed: Timed out waiting for login confirmation (e.g., Logout button).")
        print("Please check your credentials or if the website structure has changed.")
        # Optionally, save a screenshot or page source for debugging
        # driver.save_screenshot("login_timeout_debug.png")
        # with open("login_timeout_page_source.html", "w", encoding="utf-8") as f:
        #     f.write(driver.page_source)
        return False
    except NoSuchElementException:
        print("Login failed: Could not find one or more login elements on the page.")
        return False
    except Exception as e:
        print(f"An unexpected error occurred during login: {e}")
        return False

def scrape_bridgebase_hands_selenium(driver, url):
    """
    Scrapes a BridgeBase MyHands page using an already logged-in Selenium driver,
    extracts rows with class 'tourney', and creates a new HTML table.
    """
    html_content = None
    try:
        print(f"Navigating to target hands URL: {url}")
        driver.get(url)
        
        print("Waiting for page content to load (JavaScript execution)...")
        # Increased wait time slightly, and using explicit wait for a tourney row
        WebDriverWait(driver, 20).until(
            EC.presence_of_all_elements_located((By.CLASS_NAME, "tourney"))
        )
        # A small fixed wait can still be helpful if content loads in stages
        time.sleep(3) 

        html_content = driver.page_source
        print("Page source fetched for hands page.")

    except TimeoutException:
        print(f"Timed out waiting for 'tourney' rows to appear on {url}.")
        print("This could mean the page didn't load correctly, or no such rows exist (e.g., invalid tourney ID).")
        # Save page source for debugging
        # with open("hands_page_timeout_debug.html", "w", encoding="utf-8") as f:
        # f.write(driver.page_source)
        return None
    except Exception as e:
        print(f"Error during Selenium navigation or page load for hands page: {e}")
        return None

    if not html_content:
        print("Failed to get HTML content for hands page using Selenium.")
        return None

    soup = BeautifulSoup(html_content, 'html.parser')
    
    tourney_rows = soup.find_all('tr', class_='tourney')

    if not tourney_rows:
        with open("selenium_debug_output_hands.html", "w", encoding="utf-8") as f:
            f.write(html_content)
        print("No rows with class 'tourney' found on the hands page.")
        print("The HTML content fetched by Selenium has been saved to 'selenium_debug_output_hands.html' for inspection.")
        return None
    
    print(f"Found {len(tourney_rows)} rows with class 'tourney'.")

    html_output_parts = [
        "<!DOCTYPE html>", "<html lang='en'>", "<head>",
        "    <meta charset='UTF-8'>",
        "    <meta name='viewport' content='width=device-width, initial-scale=1.0'>",
        "    <title>Extracted BridgeBase Hands</title>",
        "    <style>",
        "        body { font-family: Arial, sans-serif; margin: 20px; background-color: #f4f4f4; color: #333; }",
        "        h1 { text-align: center; color: #333; }",
        "        table { border-collapse: collapse; width: 95%; margin: 20px auto; box-shadow: 0 0 15px rgba(0,0,0,0.15); background-color: #fff; }",
        "        th, td { border: 1px solid #ddd; padding: 12px; text-align: left; }",
        "        th { background-color: #007bff; color: white; font-weight: bold; }",
        "        tr:nth-child(even) { background-color: #f9f9f9; }",
        "        tr:hover { background-color: #f1f1f1; }",
        "        td a { color: #007bff; text-decoration: none; }",
        "        td a:hover { text-decoration: underline; }",
        "    </style>", "</head>", "<body>",
        "    <h1>Extracted Tournament Hands</h1>", "    <table>"
    ]
    
    for row in tourney_rows:
        html_output_parts.append(str(row)) 

    html_output_parts.extend(["    </table>", "</body>", "</html>"])
    
    return "\n".join(html_output_parts)

if __name__ == "__main__":
    #bbo_username = input("Enter your Bridge Base Online Username: ")
    #bbo_password = getpass.getpass("Enter your Bridge Base Online Password: ") # Hides password input
    bbo_username = "iit_bombay"
    bbo_password = "fin@nce"
    target_hands_url = "https://www.bridgebase.com/myhands/hands.php?tourney=SB:iit_bombay-1747543952-&username=iit_bombay"
    
    # Setup Chrome options
    chrome_options = Options()
    # chrome_options.add_argument("--headless")  # You might want to comment this out during initial testing to see the browser
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--start-maximized") # Ensure full viewport
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36")

    driver = None
    try:
        service = Service(ChromeDriverManager().install())
        driver = webdriver.Chrome(service=service, options=chrome_options)

        if login_to_bridgebase(driver, bbo_username, bbo_password):
            print(f"\nScraping data from: {target_hands_url}")
            new_html_table = scrape_bridgebase_hands_selenium(driver, target_hands_url)

            if new_html_table:
                output_filename = "extracted_bridge_hands_loggedin.html"
                try:
                    with open(output_filename, "w", encoding="utf-8") as f:
                        f.write(new_html_table)
                    print(f"Successfully extracted data and saved to {output_filename}")
                    print(f"You can open '{output_filename}' in your web browser.")
                except IOError as e:
                    print(f"Error writing to file {output_filename}: {e}")
            else:
                print("Failed to generate HTML table from the hands page.")
        else:
            print("Could not proceed with scraping due to login failure.")

    except Exception as e:
        print(f"An overall script error occurred: {e}")
    finally:
        if driver:
            driver.quit()
            print("WebDriver closed.")

103.78s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


  Using cached trio_websocket-0.12.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl.metadata (10 kB)
  Using cached outcome-1.3.0.post0-py2.py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 3.5 MB/s  0:00:02 eta 0:00:01
Using cached trio_websocket-0.12.2-py3-none-any.whl (21 kB)
Using cached outcome-1.3.0.post0-py2.py3-none-any.whl (10 kB)
Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl (29 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [selenium]5/6 [selenium]


KeyboardInterrupt: 

In [ ]:
import time
import getpass # For securely getting password
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from bs4 import BeautifulSoup

def scrape_bridgebase_hands_selenium(driver, url):
    """
    Scrapes a BridgeBase MyHands page using an already logged-in Selenium driver,
    extracts rows with class 'tourney', and creates a new HTML table.
    """
    html_content = None
    try:
        print(f"Navigating to target hands URL: {url}")
        driver.get(url)
        
        print("Waiting for page content to load (JavaScript execution)...")
        # Increased wait time slightly, and using explicit wait for a tourney row
        WebDriverWait(driver, 20).until(
            EC.presence_of_all_elements_located((By.CLASS_NAME, "tourney"))
        )
        # A small fixed wait can still be helpful if content loads in stages
        time.sleep(3) 

        html_content = driver.page_source
        print("Page source fetched for hands page.")

    except TimeoutException:
        print(f"Timed out waiting for 'tourney' rows to appear on {url}.")
        print("This could mean the page didn't load correctly, or no such rows exist (e.g., invalid tourney ID).")
        # Save page source for debugging
        # with open("hands_page_timeout_debug.html", "w", encoding="utf-8") as f:
        # f.write(driver.page_source)
        return None
    except Exception as e:
        print(f"Error during Selenium navigation or page load for hands page: {e}")
        return None

    if not html_content:
        print("Failed to get HTML content for hands page using Selenium.")
        return None

    soup = BeautifulSoup(html_content, 'html.parser')
    
    tourney_rows = soup.find_all('tr', class_='tourney')

    if not tourney_rows:
        with open("selenium_debug_output_hands.html", "w", encoding="utf-8") as f:
            f.write(html_content)
        print("No rows with class 'tourney' found on the hands page.")
        print("The HTML content fetched by Selenium has been saved to 'selenium_debug_output_hands.html' for inspection.")
        return None
    
    print(f"Found {len(tourney_rows)} rows with class 'tourney'.")

    html_output_parts = [
        "<!DOCTYPE html>", "<html lang='en'>", "<head>",
        "    <meta charset='UTF-8'>",
        "    <meta name='viewport' content='width=device-width, initial-scale=1.0'>",
        "    <title>Extracted BridgeBase Hands</title>",
        "    <style>",
        "        body { font-family: Arial, sans-serif; margin: 20px; background-color: #f4f4f4; color: #333; }",
        "        h1 { text-align: center; color: #333; }",
        "        table { border-collapse: collapse; width: 95%; margin: 20px auto; box-shadow: 0 0 15px rgba(0,0,0,0.15); background-color: #fff; }",
        "        th, td { border: 1px solid #ddd; padding: 12px; text-align: left; }",
        "        th { background-color: #007bff; color: white; font-weight: bold; }",
        "        tr:nth-child(even) { background-color: #f9f9f9; }",
        "        tr:hover { background-color: #f1f1f1; }",
        "        td a { color: #007bff; text-decoration: none; }",
        "        td a:hover { text-decoration: underline; }",
        "    </style>", "</head>", "<body>",
        "    <h1>Extracted Tournament Hands</h1>", "    <table>"
    ]
    
    for row in tourney_rows:
        html_output_parts.append(str(row)) 

    html_output_parts.extend(["    </table>", "</body>", "</html>"])
    
    return "\n".join(html_output_parts)

if __name__ == "__main__":

    driver = None
    try:
        service = Service(ChromeDriverManager().install())
        driver = webdriver.Chrome(service=service, options=chrome_options)

        if login_to_bridgebase(driver, bbo_username, bbo_password):
            print(f"\nScraping data from: {target_hands_url}")
            new_html_table = scrape_bridgebase_hands_selenium(driver, target_hands_url)

            if new_html_table:
                output_filename = "extracted_bridge_hands_loggedin.html"
                try:
                    with open(output_filename, "w", encoding="utf-8") as f:
                        f.write(new_html_table)
                    print(f"Successfully extracted data and saved to {output_filename}")
                    print(f"You can open '{output_filename}' in your web browser.")
                except IOError as e:
                    print(f"Error writing to file {output_filename}: {e}")
            else:
                print("Failed to generate HTML table from the hands page.")
        else:
            print("Could not proceed with scraping due to login failure.")

    except Exception as e:
        print(f"An overall script error occurred: {e}")
    finally:
        if driver:
            driver.quit()
            print("WebDriver closed.")

In [ ]:
page = '<table style="margin-left: auto;margin-right: auto;"><tbody><tr style="background-color: #C0CCEE"><th>Player</th>
<th>Board</th>
<th>Time</th>
<th>Result</th>
<th>Points</th>
<th>Score</th>
<th>Movie</th>
<th>Traveller</th>
</tr><tr style="background-color: #E0F0F0 "><td>iit_bombay</td><td>1</td><td>12:53</td><td>1NS=</td><td><span style="color:black; font-weight:bold">90</span></td><td><span style="color:black; font-weight:bold">61.38%</span></td><td><a target="_blank" href="http://www.bridgebase.com/tools/handviewer.html?bbo=y&amp;lin=st%7C%7Cmd%7C3SA654H86DA85CAQ85%2CSK92HKQT7DQT74CKJ%2CSQJ3HAJ52D963CT62%2CST87H943DKJ2C9743%7Csv%7C0%7Cah%7CBoard%201%7Cmb%7CP%7Cmb%7CP%7Cmb%7C1N%7Can%7Cnotrump%20opener.%20Could%20have%205M.%20--%202-5%20%21C%3B%202-5%20%21D%3B%202-5%20%21H%3B%202-5%20%21S%3B%2015-17%20HCP%3B%2018-%20total%20points%7Cmb%7CP%7Cmb%7CP%7Cmb%7CP%7Cpc%7CD4%7Cpc%7CD3%7Cpc%7CDK%7Cpc%7CD5%7Cpc%7CC9%7Cpc%7CC5%7Cpc%7CCJ%7Cpc%7CC2%7Cpc%7CHK%7Cpc%7CHA%7Cpc%7CH3%7Cpc%7CH6%7Cpc%7CC6%7Cpc%7CC3%7Cpc%7CCQ%7Cpc%7CCK%7Cpc%7CHQ%7Cpc%7CH2%7Cpc%7CH9%7Cpc%7CH8%7Cpc%7CHT%7Cpc%7CHJ%7Cpc%7CH4%7Cpc%7CS4%7Cpc%7CSQ%7Cpc%7CS7%7Cpc%7CS5%7Cpc%7CSK%7Cpc%7CH7%7Cpc%7CH5%7Cpc%7CC7%7Cpc%7CD8%7Cpc%7CS9%7Cpc%7CSJ%7Cpc%7CST%7Cpc%7CS6%7Cpc%7CS3%7Cpc%7CS8%7Cpc%7CSA%7Cpc%7CS2%7Cpc%7CCA%7Cpc%7CD7%7Cpc%7CCT%7Cpc%7CC4%7Cpc%7CDA%7Cpc%7CDT%7Cpc%7CD6%7Cpc%7CD2%7Cpc%7CC8%7Cpc%7CDQ%7Cpc%7CD9%7Cpc%7CDJ%7C">Movie</a></td><td><a target="_blank" href="/v2/daylong_hands.php?tourney=ARDARD%3A9176a9d2.3853.11f0.bb34.ac1f6b5a106e-1748059202-&amp;username=iit_bombay&amp;board=INSTANCE-T40619095-R1-B1-I106">Traveller</a></td></tr>
<tr style="background-color: #E0E0E0 "><td>iit_bombay</td><td>2</td><td>12:55</td><td>3NS-1</td><td><span style="color:black; font-weight:bold">-100</span></td><td><span style="color:black; font-weight:bold">63.48%</span></td><td><a target="_blank" href="http://www.bridgebase.com/tools/handviewer.html?bbo=y&amp;lin=st%7C%7Cmd%7C4SAJ75HQJ4DQ9CAQJ8%2CSQ82HA973DAK52C74%2CSK4HK862D83CK6532%2CST963HT5DJT764CT9%7Csv%7CN%7Cah%7CBoard%202%7Cmb%7CP%7Cmb%7C1N%7Can%7Cnotrump%20opener.%20Could%20have%205M.%20--%202-5%20%21C%3B%202-5%20%21D%3B%202-5%20%21H%3B%202-5%20%21S%3B%2015-17%20HCP%3B%2018-%20total%20points%7Cmb%7CP%7Cmb%7C2C%7Can%7CStayman%20--%20%20%7Cmb%7CP%7Cmb%7C2S%7Can%7C2-5%20%21C%3B%202-5%20%21D%3B%202-3%20%21H%3B%204-5%20%21S%3B%2015-17%20HCP%3B%2018-%20total%20points%7Cmb%7CP%7Cmb%7C2N%7Can%7CInvite%20to%203NT%2C%20may%20not%20have%204-card%20major%20--%204-%20%21H%3B%203-%20%21S%3B%209%20HCP%7Cmb%7CP%7Cmb%7C3N%7Can%7C2-5%20%21C%3B%202-5%20%21D%3B%202-3%20%21H%3B%204-5%20%21S%3B%2016-17%20HCP%3B%2018-%20total%20points%7Cmb%7CP%7Cmb%7CP%7Cmb%7CP%7Cpc%7CH3%7Cpc%7CH2%7Cpc%7CH5%7Cpc%7CHQ%7Cpc%7CCA%7Cpc%7CC7%7Cpc%7CC2%7Cpc%7CC9%7Cpc%7CCQ%7Cpc%7CC4%7Cpc%7CC3%7Cpc%7CCT%7Cpc%7CCJ%7Cpc%7CD5%7Cpc%7CC5%7Cpc%7CD6%7Cpc%7CC8%7Cpc%7CH7%7Cpc%7CCK%7Cpc%7CHT%7Cpc%7CC6%7Cpc%7CD7%7Cpc%7CH4%7Cpc%7CH9%7Cpc%7CSK%7Cpc%7CS6%7Cpc%7CS5%7Cpc%7CS2%7Cpc%7CS4%7Cpc%7CS3%7Cpc%7CSA%7Cpc%7CS8%7Cpc%7CHJ%7Cpc%7CHA%7Cpc%7CH6%7Cpc%7CD4%7Cpc%7CSQ%7Cpc%7CH8%7Cpc%7CS9%7Cpc%7CS7%7Cpc%7CDK%7Cpc%7CD3%7Cpc%7CDJ%7Cpc%7CD9%7Cpc%7CDA%7Cpc%7CD8%7Cpc%7CDT%7Cpc%7CDQ%7Cpc%7CD2%7Cpc%7CHK%7Cpc%7CST%7Cpc%7CSJ%7C">Movie</a></td><td><a target="_blank" href="/v2/daylong_hands.php?tourney=ARDARD%3A9176a9d2.3853.11f0.bb34.ac1f6b5a106e-1748059202-&amp;username=iit_bombay&amp;board=INSTANCE-T40619095-R1-B2-I155">Traveller</a></td></tr>
<tr style="background-color: #E0F0F0 "><td>iit_bombay</td><td>3</td><td>12:57</td><td>4<span style="color:red">♥</span>S=</td><td><span style="color:black; font-weight:bold">420</span></td><td><span style="color:black; font-weight:bold">94.55%</span></td><td><a target="_blank" href="http://www.bridgebase.com/tools/handviewer.html?bbo=y&amp;lin=st%7C%7Cmd%7C1SJT8532HKQT6DKCAQ%2CSKQ64HAJ7D4CT7643%2CSH9543DAT9762CKJ9%2CSA97H82DQJ853C852%7Csv%7CE%7Cah%7CBoard%203%7Cmb%7C1S%7Can%7CMajor%20suit%20opening%20--%205%2B%20%21S%3B%2011-21%20HCP%3B%2012-22%20total%20points%7Cmb%7CP%7Cmb%7C1N%7Can%7CForcing%20one%20notrump%20--%203-%20%21S%3B%206%2B%20HCP%3B%2012-%20total%20points%7Cmb%7CP%7Cmb%7C2H%7Can%7CNew%20suit%20--%204%2B%20%21H%3B%205%2B%20%21S%3B%2011%2B%20HCP%3B%2012-18%20total%20points%7Cmb%7CP%7Cmb%7C4H%7Can%7C4%2B%20%21H%3B%203-%20%21S%3B%2011-12%20total%20points%7Cmb%7CP%7Cmb%7CP%7Cmb%7CP%7Cpc%7CD4%7Cpc%7CD2%7Cpc%7CD5%7Cpc%7CDK%7Cpc%7CS2%7Cpc%7CS6%7Cpc%7CH3%7Cpc%7CS7%7Cpc%7CDA%7Cpc%7CD3%7Cpc%7CS3%7Cpc%7CH7%7Cpc%7CC3%7Cpc%7CC9%7Cpc%7CC2%7Cpc%7CCQ%7Cpc%7CS5%7Cpc%7CS4%7Cpc%7CH4%7Cpc%7CS9%7Cpc%7CCJ%7Cpc%7CC8%7Cpc%7CCA%7Cpc%7CCT%7Cpc%7CS8%7Cpc%7CSQ%7Cpc%7CH5%7Cpc%7CSA%7Cpc%7CH9%7Cpc%7CH8%7Cpc%7CHQ%7Cpc%7CHA%7Cpc%7CSK%7Cpc%7CD6%7Cpc%7CC5%7Cpc%7CST%7Cpc%7CC6%7Cpc%7CCK%7Cpc%7CH2%7Cpc%7CH6%7Cpc%7CHK%7Cpc%7CHJ%7Cpc%7CD7%7Cpc%7CD8%7Cpc%7CHT%7Cpc%7CC7%7Cpc%7CD9%7Cpc%7CDJ%7Cpc%7CSJ%7Cpc%7CC4%7Cpc%7CDT%7Cpc%7CDQ%7C">Movie</a></td><td><a target="_blank" href="/v2/daylong_hands.php?tourney=ARDARD%3A9176a9d2.3853.11f0.bb34.ac1f6b5a106e-1748059202-&amp;username=iit_bombay&amp;board=INSTANCE-T40619095-R1-B3-I129">Traveller</a></td></tr>
<tr style="background-color: #E0E0E0 "><td>iit_bombay</td><td>4</td><td>12:59</td><td>4<span style="color:red">♥</span>S-1</td><td><span style="color:black; font-weight:bold">-100</span></td><td><span style="color:black; font-weight:bold">54.37%</span></td><td><a target="_blank" href="http://www.bridgebase.com/tools/handviewer.html?bbo=y&amp;lin=st%7C%7Cmd%7C2S82HAKQ975DKT954C%2CSK9643HDA862CJ432%2CSAQJT7HJ4D73CAT75%2CS5HT8632DQJCKQ986%7Csv%7CB%7Cah%7CBoard%204%7Cmb%7CP%7Cmb%7C1S%7Can%7CMajor%20suit%20opening%20--%205%2B%20%21S%3B%2011-21%20HCP%3B%2012-22%20total%20points%7Cmb%7CP%7Cmb%7C2H%7Can%7CForcing%20two%20over%20one%20--%205%2B%20%21H%3B%2012%2B%20HCP%3B%2013%2B%20total%20points%3B%20forcing%20to%203N%7Cmb%7CP%7Cmb%7C2S%7Can%7COpener%20rebids%20suit%20--%203-%20%21H%3B%205%2B%20%21S%3B%2011-21%20HCP%3B%2012-22%20total%20points%3B%20forcing%20to%203N%7Cmb%7CP%7Cmb%7C4H%7Can%7C12%2B%20HCP%3B%20strong%20rebiddable%20%21H%3B%2013%2B%20total%20points%7Cmb%7CP%7Cmb%7CP%7Cmb%7CP%7Cpc%7CC2%7Cpc%7CCA%7Cpc%7CC6%7Cpc%7CD4%7Cpc%7CH4%7Cpc%7CH3%7Cpc%7CHA%7Cpc%7CS3%7Cpc%7CS2%7Cpc%7CS4%7Cpc%7CST%7Cpc%7CS5%7Cpc%7CHJ%7Cpc%7CH2%7Cpc%7CH5%7Cpc%7CC3%7Cpc%7CC5%7Cpc%7CC8%7Cpc%7CH7%7Cpc%7CC4%7Cpc%7CHK%7Cpc%7CD8%7Cpc%7CC7%7Cpc%7CH8%7Cpc%7CHQ%7Cpc%7CD2%7Cpc%7CCT%7Cpc%7CH6%7Cpc%7CS8%7Cpc%7CS9%7Cpc%7CSJ%7Cpc%7CC9%7Cpc%7CSA%7Cpc%7CHT%7Cpc%7CD5%7Cpc%7CS6%7Cpc%7CCK%7Cpc%7CH9%7Cpc%7CCJ%7Cpc%7CD3%7Cpc%7CD9%7Cpc%7CD6%7Cpc%7CD7%7Cpc%7CDJ%7Cpc%7CCQ%7Cpc%7CDT%7Cpc%7CSK%7Cpc%7CS7%7Cpc%7CDQ%7Cpc%7CDK%7Cpc%7CDA%7Cpc%7CSQ%7C">Movie</a></td><td><a target="_blank" href="/v2/daylong_hands.php?tourney=ARDARD%3A9176a9d2.3853.11f0.bb34.ac1f6b5a106e-1748059202-&amp;username=iit_bombay&amp;board=INSTANCE-T40619095-R1-B4-I190">Traveller</a></td></tr>
<tr style="background-color: #E0F0F0 "><td>iit_bombay</td><td>5</td><td>13:01</td><td>4<span style="color:red">♦</span>W-1</td><td><span style="color:black; font-weight:bold">50</span></td><td><span style="color:black; font-weight:bold">58.25%</span></td><td><a target="_blank" href="http://www.bridgebase.com/tools/handviewer.html?bbo=y&amp;lin=st%7C%7Cmd%7C3SAT5HKQT983D6CAJT%2CS94HA42DAJ854CK92%2CSQJ873HJ7D92C7654%2CSK62H65DKQT73CQ83%7Csv%7CN%7Cah%7CBoard%205%7Cmb%7CP%7Cmb%7CP%7Cmb%7C1H%7Can%7CMajor%20suit%20opening%20--%205%2B%20%21H%3B%2011-21%20HCP%3B%2012-22%20total%20points%7Cmb%7C2D%7Can%7CTwo-level%20overcall%20--%205%2B%20%21D%3B%2010%2B%20HCP%3B%2011-18%20total%20points%7Cmb%7CP%7Cmb%7C2H%7Can%7CGood%20support%20in%20D%20--%203%2B%20%21D%3B%2011-%20HCP%3B%2011-12%20total%20points%7Cmb%7C3H%7Can%7C6%2B%20%21H%3B%2021-%20HCP%3B%2015-22%20total%20points%7Cmb%7CP%7Cmb%7CP%7Cmb%7C4D%7Can%7C5%2B%20%21D%3B%2011-%20HCP%3B%2011-12%20total%20points%7Cmb%7CP%7Cmb%7CP%7Cmb%7CP%7Cpc%7CHJ%7Cpc%7CH5%7Cpc%7CH8%7Cpc%7CHA%7Cpc%7CD8%7Cpc%7CD2%7Cpc%7CDT%7Cpc%7CD6%7Cpc%7CD3%7Cpc%7CH9%7Cpc%7CDJ%7Cpc%7CD9%7Cpc%7CH2%7Cpc%7CH7%7Cpc%7CH6%7Cpc%7CH3%7Cpc%7CC6%7Cpc%7CC3%7Cpc%7CCT%7Cpc%7CCK%7Cpc%7CH4%7Cpc%7CS3%7Cpc%7CDK%7Cpc%7CHT%7Cpc%7CDQ%7Cpc%7CHQ%7Cpc%7CDA%7Cpc%7CC4%7Cpc%7CC2%7Cpc%7CC7%7Cpc%7CC8%7Cpc%7CCJ%7Cpc%7CCA%7Cpc%7CC9%7Cpc%7CC5%7Cpc%7CCQ%7Cpc%7CSA%7Cpc%7CS4%7Cpc%7CS7%7Cpc%7CS6%7Cpc%7CS5%7Cpc%7CS9%7Cpc%7CS8%7Cpc%7CS2%7Cpc%7CD4%7Cpc%7CSJ%7Cpc%7CD7%7Cpc%7CST%7Cpc%7CSK%7Cpc%7CHK%7Cpc%7CD5%7Cpc%7CSQ%7C">Movie</a></td><td><a target="_blank" href="/v2/daylong_hands.php?tourney=ARDARD%3A9176a9d2.3853.11f0.bb34.ac1f6b5a106e-1748059202-&amp;username=iit_bombay&amp;board=INSTANCE-T40619095-R1-B5-I150">Traveller</a></td></tr>
<tr style="background-color: #E0E0E0 "><td>iit_bombay</td><td>6</td><td>13:02</td><td>4<span style="color:black">♠</span>S+1</td><td><span style="color:red; font-weight:bold">450</span></td><td><span style="color:red; font-weight:bold">47.09%</span></td><td><a target="_blank" href="http://www.bridgebase.com/tools/handviewer.html?bbo=y&amp;lin=st%7C%7Cmd%7C4SAQJT942HQ43DAKC7%2CS87HT872DQ5CAT865%2CS63HA95DJT98764CQ%2CSK5HKJ6D32CKJ9432%7Csv%7CE%7Cah%7CBoard%206%7Cmb%7CP%7Cmb%7C1S%7Can%7CMajor%20suit%20opening%20--%205%2B%20%21S%3B%2011-21%20HCP%3B%2012-22%20total%20points%7Cmb%7CP%7Cmb%7C1N%7Can%7CForcing%20one%20notrump%20--%203-%20%21S%3B%206%2B%20HCP%3B%2012-%20total%20points%7Cmb%7CP%7Cmb%7C4S%7Can%7C7%2B%20%21S%3B%2017%2B%20HCP%3B%2018-21%20total%20points%7Cmb%7CP%7Cmb%7CP%7Cmb%7CP%7Cpc%7CCA%7Cpc%7CCQ%7Cpc%7CC9%7Cpc%7CC7%7Cpc%7CH2%7Cpc%7CH5%7Cpc%7CHK%7Cpc%7CH3%7Cpc%7CSK%7Cpc%7CSA%7Cpc%7CS7%7Cpc%7CS3%7Cpc%7CSQ%7Cpc%7CS8%7Cpc%7CS6%7Cpc%7CS5%7Cpc%7CSJ%7Cpc%7CH7%7Cpc%7CD4%7Cpc%7CC3%7Cmc%7C11%7C">Movie</a></td><td><a target="_blank" href="/v2/daylong_hands.php?tourney=ARDARD%3A9176a9d2.3853.11f0.bb34.ac1f6b5a106e-1748059202-&amp;username=iit_bombay&amp;board=INSTANCE-T40619095-R1-B6-I127">Traveller</a></td></tr>
<tr style="background-color: #E0F0F0 "><td>iit_bombay</td><td>7</td><td>13:03</td><td>4<span style="color:red">♥</span>W=</td><td><span style="color:black; font-weight:bold">-620</span></td><td><span style="color:black; font-weight:bold">60.89%</span></td><td><a target="_blank" href="http://www.bridgebase.com/tools/handviewer.html?bbo=y&amp;lin=st%7C%7Cmd%7C1SAQT964HJDKQ5CA43%2CSKJ52HAQ984DJ7C72%2CS83H65DT86432CQT5%2CS7HKT732DA9CKJ986%7Csv%7CB%7Cah%7CBoard%207%7Cmb%7C1S%7Can%7CMajor%20suit%20opening%20--%205%2B%20%21S%3B%2011-21%20HCP%3B%2012-22%20total%20points%7Cmb%7C2H%7Can%7CTwo-level%20overcall%20--%205%2B%20%21H%3B%2010%2B%20HCP%3B%2011-18%20total%20points%7Cmb%7CP%7Cmb%7C2S%7Can%7CGood%20support%20in%20H%20--%203%2B%20%21H%3B%2011%2B%20total%20points%7Cmb%7C3S%7Can%7C6%2B%20%21S%3B%2021-%20HCP%3B%2015-22%20total%20points%7Cmb%7CP%7Cmb%7CP%7Cmb%7C4H%7Can%7C3%2B%20%21H%3B%2013%2B%20HCP%3B%2014-20%20total%20points%7Cmb%7CP%7Cmb%7CP%7Cmb%7CP%7Cpc%7CD4%7Cpc%7CD9%7Cpc%7CDQ%7Cpc%7CD7%7Cpc%7CSA%7Cpc%7CS5%7Cpc%7CS8%7Cpc%7CS7%7Cpc%7CCA%7Cpc%7CC2%7Cpc%7CC5%7Cpc%7CC8%7Cpc%7CD5%7Cpc%7CDJ%7Cpc%7CDT%7Cpc%7CDA%7Cpc%7CH2%7Cmc%7C10%7C">Movie</a></td><td><a target="_blank" href="/v2/daylong_hands.php?tourney=ARDARD%3A9176a9d2.3853.11f0.bb34.ac1f6b5a106e-1748059202-&amp;username=iit_bombay&amp;board=INSTANCE-T40619095-R1-B7-I87">Traveller</a></td></tr>
<tr style="background-color: #E0E0E0 "><td>iit_bombay</td><td>8</td><td>13:05</td><td>1NN+3</td><td><span style="color:black; font-weight:bold">180</span></td><td><span style="color:black; font-weight:bold">58.93%</span></td><td><a target="_blank" href="http://www.bridgebase.com/tools/handviewer.html?bbo=y&amp;lin=st%7C%7Cmd%7C2SAKJ98HA52D62CJT5%2CS762H8764DK7CAQ76%2CS53HKQJ9DAT43C843%2CSQT4HT3DQJ985CK92%7Csv%7C0%7Cah%7CBoard%208%7Cmb%7CP%7Cmb%7CP%7Cmb%7CP%7Cmb%7C1S%7Can%7CMajor%20suit%20opening%20--%205%2B%20%21S%3B%2011-21%20HCP%3B%2012-22%20total%20points%7Cmb%7CP%7Cmb%7C1N%7Can%7C2-%20%21S%3B%206-11%20HCP%3B%2012-%20total%20points%7Cmb%7CP%7Cmb%7CP%7Cmb%7CP%7Cpc%7CD9%7Cpc%7CD2%7Cpc%7CDK%7Cpc%7CDA%7Cpc%7CS3%7Cpc%7CS4%7Cpc%7CSJ%7Cpc%7CS2%7Cpc%7CSA%7Cpc%7CS6%7Cpc%7CS5%7Cpc%7CSQ%7Cpc%7CSK%7Cpc%7CS7%7Cpc%7CC3%7Cpc%7CST%7Cpc%7CS9%7Cpc%7CH8%7Cpc%7CC4%7Cpc%7CC2%7Cpc%7CS8%7Cpc%7CC7%7Cpc%7CC8%7Cpc%7CH3%7Cpc%7CHA%7Cpc%7CH6%7Cpc%7CH9%7Cpc%7CHT%7Cpc%7CH2%7Cpc%7CH4%7Cpc%7CHJ%7Cpc%7CD8%7Cpc%7CHK%7Cpc%7CD5%7Cpc%7CH5%7Cpc%7CH7%7Cpc%7CHQ%7Cpc%7CC9%7Cpc%7CD6%7Cpc%7CC6%7Cpc%7CD4%7Cpc%7CDJ%7Cpc%7CC5%7Cpc%7CD7%7Cpc%7CCK%7Cpc%7CCT%7Cpc%7CCA%7Cpc%7CD3%7Cpc%7CCQ%7Cpc%7CDT%7Cpc%7CDQ%7Cpc%7CCJ%7C">Movie</a></td><td><a target="_blank" href="/v2/daylong_hands.php?tourney=ARDARD%3A9176a9d2.3853.11f0.bb34.ac1f6b5a106e-1748059202-&amp;username=iit_bombay&amp;board=INSTANCE-T40619095-R1-B8-I110">Traveller</a></td></tr>
<tr style="background-color: #E0F0F0 "><td>iit_bombay</td><td>9</td><td>13:07</td><td>2<span style="color:red">♥</span>S+1</td><td><span style="color:black; font-weight:bold">140</span></td><td><span style="color:black; font-weight:bold">79.35%</span></td><td><a target="_blank" href="http://www.bridgebase.com/tools/handviewer.html?bbo=y&amp;lin=st%7C%7Cmd%7C3SA65HKQ9865DK6CKT%2CSQ8HAT2DQT97CJ963%2CSJ72H74D832CAQ854%2CSKT943HJ3DAJ54C72%7Csv%7CE%7Cah%7CBoard%209%7Cmb%7CP%7Cmb%7CP%7Cmb%7C1H%7Can%7CMajor%20suit%20opening%20--%205%2B%20%21H%3B%2011-21%20HCP%3B%2012-22%20total%20points%7Cmb%7CP%7Cmb%7C1N%7Can%7C2-%20%21H%3B%206-11%20HCP%3B%2012-%20total%20points%7Cmb%7CP%7Cmb%7C2H%7Can%7C6%2B%20%21H%3B%2011%2B%20HCP%3B%2012-16%20total%20points%7Cmb%7CP%7Cmb%7CP%7Cmb%7CP%7Cpc%7CC3%7Cpc%7CC4%7Cpc%7CC7%7Cpc%7CCT%7Cpc%7CHQ%7Cpc%7CH2%7Cpc%7CH4%7Cpc%7CH3%7Cpc%7CH5%7Cpc%7CHT%7Cpc%7CH7%7Cpc%7CHJ%7Cpc%7CDA%7Cpc%7CD6%7Cpc%7CDT%7Cpc%7CD2%7Cpc%7CS4%7Cpc%7CS5%7Cpc%7CSQ%7Cpc%7CS2%7Cpc%7CD9%7Cpc%7CD3%7Cpc%7CD4%7Cpc%7CDK%7Cpc%7CHK%7Cpc%7CHA%7Cpc%7CD8%7Cpc%7CST%7Cpc%7CDQ%7Cpc%7CC5%7Cpc%7CD5%7Cpc%7CH6%7Cpc%7CH9%7Cpc%7CD7%7Cpc%7CS7%7Cpc%7CC2%7Cpc%7CCK%7Cpc%7CC6%7Cpc%7CCA%7Cpc%7CS3%7Cpc%7CCQ%7Cpc%7CS9%7Cpc%7CS6%7Cpc%7CCJ%7Cpc%7CSJ%7Cpc%7CSK%7Cpc%7CSA%7Cpc%7CS8%7Cpc%7CH8%7Cpc%7CC9%7Cpc%7CC8%7Cpc%7CDJ%7C">Movie</a></td><td><a target="_blank" href="/v2/daylong_hands.php?tourney=ARDARD%3A9176a9d2.3853.11f0.bb34.ac1f6b5a106e-1748059202-&amp;username=iit_bombay&amp;board=INSTANCE-T40619095-R1-B9-I102">Traveller</a></td></tr>
<tr style="background-color: #E0E0E0 "><td>iit_bombay</td><td>10</td><td>13:09</td><td>3NN-1</td><td><span style="color:black; font-weight:bold">-100</span></td><td><span style="color:black; font-weight:bold">50.00%</span></td><td><a target="_blank" href="http://www.bridgebase.com/tools/handviewer.html?bbo=y&amp;lin=st%7C%7Cmd%7C4SKJ86HADJ6543CAK4%2CSA2HQ864DT8CQ9752%2CSQ75HKT532DQ2CJT3%2CST943HJ97DAK97C86%7Csv%7CB%7Cah%7CBoard%2010%7Cmb%7CP%7Cmb%7C1D%7Can%7CMinor%20suit%20opening%20--%203%2B%20%21D%3B%2011-21%20HCP%3B%2012-22%20total%20points%7Cmb%7CP%7Cmb%7C1H%7Can%7COne%20over%20one%20--%204%2B%20%21H%3B%206%2B%20total%20points%7Cmb%7CP%7Cmb%7C1S%7Can%7C4%2B%20%21D%3B%203-%20%21H%3B%204%2B%20%21S%3B%2011%2B%20HCP%3B%2012-18%20total%20points%7Cmb%7CP%7Cmb%7C1N%7Can%7CBalanced%20minimum%20--%202%2B%20%21C%3B%202-3%20%21D%3B%204%2B%20%21H%3B%202-3%20%21S%3B%206-10%20HCP%7Cmb%7CP%7Cmb%7C3N%7Can%7C4%2B%20%21D%3B%203-%20%21H%3B%204%20%21S%3B%2018%2B%20HCP%3B%2018-%20total%20points%7Cmb%7CP%7Cmb%7CP%7Cmb%7CP%7Cpc%7CC8%7Cpc%7CC4%7Cpc%7CCQ%7Cpc%7CC3%7Cpc%7CH6%7Cpc%7CH2%7Cpc%7CH7%7Cpc%7CHA%7Cpc%7CD3%7Cpc%7CD8%7Cpc%7CDQ%7Cpc%7CDK%7Cpc%7CHJ%7Cpc%7CD4%7Cpc%7CH4%7Cpc%7CHK%7Cpc%7CS5%7Cpc%7CS4%7Cpc%7CSJ%7Cpc%7CSA%7Cpc%7CHQ%7Cpc%7CH3%7Cpc%7CH9%7Cpc%7CD5%7Cpc%7CDT%7Cpc%7CD2%7Cpc%7CDA%7Cpc%7CD6%7Cpc%7CD9%7Cpc%7CDJ%7Cpc%7CC5%7Cpc%7CCT%7Cpc%7CCA%7Cpc%7CC9%7Cpc%7CCJ%7Cpc%7CC6%7Cpc%7CCK%7Cpc%7CC7%7Cpc%7CH5%7Cpc%7CD7%7Cpc%7CS6%7Cpc%7CS2%7Cpc%7CSQ%7Cpc%7CS3%7Cpc%7CHT%7Cpc%7CS9%7Cpc%7CS8%7Cpc%7CH8%7Cpc%7CS7%7Cpc%7CST%7Cpc%7CSK%7Cpc%7CC2%7C">Movie</a></td><td><a target="_blank" href="/v2/daylong_hands.php?tourney=ARDARD%3A9176a9d2.3853.11f0.bb34.ac1f6b5a106e-1748059202-&amp;username=iit_bombay&amp;board=INSTANCE-T40619095-R1-B10-I108">Traveller</a></td></tr>
</tbody></table>'

In [ ]:
# get the rendered page

import requests
from bs4 import BeautifulSoup

def scrape_bridgebase_hands(url):
    """
    Scrapes a BridgeBase MyHands page, extracts rows with class 'tourney',
    and creates a new HTML table from them.
    """
    try:
        # Add a User-Agent header to mimic a browser request
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        response = requests.get(url, headers=headers)
        response.raise_for_status()  # Raise an exception for bad status codes (4xx or 5xx)
        html_content = response.text
    except requests.exceptions.RequestException as e:
        print(f"Error fetching URL: {e}")
        return None

    soup = BeautifulSoup(html_content, 'html.parser')
    
    # Find all <tr> elements with class="tourney"
    # Note: 'class_' is used because 'class' is a Python keyword
    tourney_rows = soup.find_all('tr', class_='tourney')

    if not tourney_rows:
        print("No rows with class 'tourney' found.")
        return None

    # Start building the new HTML output
    html_output_parts = [
        "<!DOCTYPE html>",
        "<html lang='en'>",
        "<head>",
        "    <meta charset='UTF-8'>",
        "    <meta name='viewport' content='width=device-width, initial-scale=1.0'>",
        "    <title>Extracted BridgeBase Hands</title>",
        "    <style>",
        "        body { font-family: Arial, sans-serif; margin: 20px; background-color: #f4f4f4; color: #333; }",
        "        h1 { text-align: center; color: #333; }",
        "        table { border-collapse: collapse; width: 95%; margin: 20px auto; box-shadow: 0 0 15px rgba(0,0,0,0.15); background-color: #fff; }",
        "        th, td { border: 1px solid #ddd; padding: 12px; text-align: left; }",
        "        th { background-color: #007bff; color: white; font-weight: bold; }",
        "        tr:nth-child(even) { background-color: #f9f9f9; }",
        "        tr:hover { background-color: #f1f1f1; }",
        "        /* Style links within the table if any */",
        "        td a { color: #007bff; text-decoration: none; }",
        "        td a:hover { text-decoration: underline; }",
        "    </style>",
        "</head>",
        "<body>",
        "    <h1>Extracted Tournament Hands</h1>",
        "    <table>"
    ]
    
    # Add a header row (optional, but good for structure)
    # You might need to inspect the original table to define appropriate headers
    # or if the first 'tourney' row acts as a header.
    # For this example, let's assume the columns are: Board, Vul, Dealer, Contract, By, Tricks, Score, N-S Names, E-W Names, Result
    # This is a generic header. The actual `tr class="tourney"` rows already contain <td> cells.
    # If the `tourney` rows themselves contain `th` for the first one, you might not need this.
    # Upon inspection, the `tourney` rows are data rows (td). A separate `<tr>` with `<th>` exists above them.
    # For simplicity as per request, we are only including `tr class="tourney"`.
    # If you want the original headers, you'd scrape them separately.
    # For example:
    # original_header_row = soup.find('table').find('thead').find('tr') # or similar logic
    # if original_header_row:
    # html_output_parts.append(str(original_header_row))

    for row in tourney_rows:
        # The 'row' object is a BeautifulSoup Tag. str(row) gives its HTML representation.
        html_output_parts.append(str(row)) 

    html_output_parts.extend([
        "    </table>",
        "</body>",
        "</html>"
    ])
    
    return "\n".join(html_output_parts)

if __name__ == "__main__":
    target_url = "https://www.bridgebase.com/myhands/hands.php?tourney=SB:iit_bombay-1747543952-&username=iit_bombay"
    
    print(f"Scraping data from: {target_url}")
    new_html_table = scrape_bridgebase_hands(target_url)

    if new_html_table:
        output_filename = "extracted_bridge_hands.html"
        try:
            with open(output_filename, "w", encoding="utf-8") as f:
                f.write(new_html_table)
            print(f"Successfully extracted data and saved to {output_filename}")
            print(f"You can open '{output_filename}' in your web browser to view the table.")
        except IOError as e:
            print(f"Error writing to file {output_filename}: {e}")
    else:
        print("Failed to generate HTML table.")

Scraping data from: https://www.bridgebase.com/myhands/hands.php?tourney=SB:iit_bombay-1747543952-&username=iit_bombay


No rows with class 'tourney' found.
Failed to generate HTML table.


In [2]:
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager # For easier driver management
from bs4 import BeautifulSoup

def scrape_bridgebase_hands_selenium(url):
    """
    Scrapes a BridgeBase MyHands page using Selenium to handle JavaScript,
    extracts rows with class 'tourney', and creates a new HTML table.
    """
    # Setup Chrome options
    chrome_options = Options()
    chrome_options.add_argument("--headless")  # Run headless (no browser window visible)
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36")

    # Use webdriver_manager to automatically download and manage ChromeDriver
    try:
        service = Service(ChromeDriverManager().install())
        driver = webdriver.Chrome(service=service, options=chrome_options)
    except Exception as e:
        print(f"Error setting up WebDriver: {e}")
        print("Please ensure ChromeDriver is installed and accessible, or webdriver_manager can download it.")
        print("You can download ChromeDriver from: https://chromedriver.chromium.org/downloads")
        return None

    html_content = None
    try:
        print(f"Navigating to {url} with Selenium...")
        driver.get(url)
        
        # Wait for JavaScript to load content.
        # You might need to adjust this time or use explicit waits
        # for specific elements to appear.
        # For this page, the content seems to load reasonably fast,
        # but a more robust solution would be an explicit wait.
        print("Waiting for page content to load (JavaScript execution)...")
        time.sleep(5) # Wait 5 seconds. Adjust if necessary.

        # (Optional but recommended for robustness: Explicit Wait)
        # from selenium.webdriver.common.by import By
        # from selenium.webdriver.support.ui import WebDriverWait
        # from selenium.webdriver.support import expected_conditions as EC
        # try:
        #     WebDriverWait(driver, 10).until(
        #         EC.presence_of_element_located((By.CLASS_NAME, "tourney"))
        #     )
        #     print("Element with class 'tourney' found.")
        # except TimeoutException:
        #     print("Timed out waiting for page to load or 'tourney' class to appear.")
        #     driver.quit()
        #     return None

        html_content = driver.page_source
        print("Page source fetched.")

    except Exception as e:
        print(f"Error during Selenium navigation or page load: {e}")
        return None
    finally:
        if driver:
            driver.quit()
            print("WebDriver closed.")

    if not html_content:
        print("Failed to get HTML content using Selenium.")
        return None

    soup = BeautifulSoup(html_content, 'html.parser')
    
    # Find all <tr> elements with class="tourney"
    tourney_rows = soup.find_all('tr', class_='tourney')

    if not tourney_rows:
        # If still not found, save the HTML Selenium got for inspection
        with open("selenium_debug_output.html", "w", encoding="utf-8") as f:
            f.write(html_content)
        print("No rows with class 'tourney' found even after using Selenium.")
        print("The HTML content fetched by Selenium has been saved to 'selenium_debug_output.html' for inspection.")
        return None
    
    print(f"Found {len(tourney_rows)} rows with class 'tourney'.")

    # Start building the new HTML output
    html_output_parts = [
        "<!DOCTYPE html>",
        "<html lang='en'>",
        "<head>",
        "    <meta charset='UTF-8'>",
        "    <meta name='viewport' content='width=device-width, initial-scale=1.0'>",
        "    <title>Extracted BridgeBase Hands</title>",
        "    <style>",
        "        body { font-family: Arial, sans-serif; margin: 20px; background-color: #f4f4f4; color: #333; }",
        "        h1 { text-align: center; color: #333; }",
        "        table { border-collapse: collapse; width: 95%; margin: 20px auto; box-shadow: 0 0 15px rgba(0,0,0,0.15); background-color: #fff; }",
        "        th, td { border: 1px solid #ddd; padding: 12px; text-align: left; }",
        "        th { background-color: #007bff; color: white; font-weight: bold; }",
        "        tr:nth-child(even) { background-color: #f9f9f9; }",
        "        tr:hover { background-color: #f1f1f1; }",
        "        td a { color: #007bff; text-decoration: none; }",
        "        td a:hover { text-decoration: underline; }",
        "    </style>",
        "</head>",
        "<body>",
        "    <h1>Extracted Tournament Hands</h1>",
        "    <table>"
    ]
    
    for row in tourney_rows:
        html_output_parts.append(str(row)) 

    html_output_parts.extend([
        "    </table>",
        "</body>",
        "</html>"
    ])
    
    return "\n".join(html_output_parts)

if __name__ == "__main__":
    target_url = "https://www.bridgebase.com/myhands/hands.php?tourney=SB:iit_bombay-1747543952-&username=iit_bombay&from_login=0"
    
    print(f"Scraping data from: {target_url}")
    new_html_table = scrape_bridgebase_hands_selenium(target_url)

    if new_html_table:
        output_filename = "extracted_bridge_hands_selenium.html"
        try:
            with open(output_filename, "w", encoding="utf-8") as f:
                f.write(new_html_table)
            print(f"Successfully extracted data and saved to {output_filename}")
            print(f"You can open '{output_filename}' in your web browser to view the table.")
        except IOError as e:
            print(f"Error writing to file {output_filename}: {e}")
    else:
        print("Failed to generate HTML table using Selenium.")

Scraping data from: https://www.bridgebase.com/myhands/hands.php?tourney=SB:iit_bombay-1747543952-&username=iit_bombay&from_login=0
Navigating to https://www.bridgebase.com/myhands/hands.php?tourney=SB:iit_bombay-1747543952-&username=iit_bombay&from_login=0 with Selenium...
Waiting for page content to load (JavaScript execution)...
Page source fetched.
WebDriver closed.
No rows with class 'tourney' found even after using Selenium.
The HTML content fetched by Selenium has been saved to 'selenium_debug_output.html' for inspection.
Failed to generate HTML table using Selenium.


In [ ]:
tr class="tourney">
		<td class="handnum">1</td>
		<td>21:54</td>
		<td class="north">GiB</td>
		<td class="south">iit_bombay</td>
		<td class="east">GiB</td>
		<td class="west">GiB</td>
		<td class="result">1NS+1</td>
		<td class="score">120</td>
		<td class="score">75.00%</td>
		<td class="movie"><A href="https://www.bridgebase.com/tools/handviewer.html?bbo=y&myhand=M-4442664572-1747543952" onclick="hv_popuplin('pn|iit_bombay,~~M5995dif,~~M60140bn,~~M6017gdr|st%7C%7Cmd%7C3S2H48KD56QKC6TQKA%2CS4JAH26JAD9JAC478%2CS789TKH37QD4TC25J%2C%7Crh%7C%7Cah%7CBoard%201%7Csv%7Co%7Cmb%7Cp%7Cmb%7Cp%7Cmb%7C1C%7Can%7CMinor%20suit%20opening%20--%203%2B%20%21C%3B%2011-21%20HCP%3B%2012-22%20total%20points%7Cmb%7Cd%7Can%7CTakeout%20double%20--%202-%20%21C%3B%203-5%20%21D%3B%203-4%20%21H%3B%203-4%20%21S%3B%2012%2B%20total%20points%20%7Cmb%7C1S%7Can%7CFree%20bid%3B%20new%20suit%20--%204%2B%20%21S%3B%2011-%20HCP%3B%208-12%20total%20points%20%7Cmb%7Cp%7Cmb%7C1N%7Can%7C3-5%20%21C%3B%202-4%20%21D%3B%202-4%20%21H%3B%202-3%20%21S%3B%2012-14%20HCP%7Cmb%7Cp%7Cmb%7Cp%7Cmb%7Cp%7Cpc%7CC4%7Cpc%7CC2%7Cpc%7CC9%7Cpc%7CCK%7Cpc%7CS2%7Cpc%7CS4%7Cpc%7CS7%7Cpc%7CSQ%7Cpc%7CD8%7Cpc%7CD5%7Cpc%7CDJ%7Cpc%7CD4%7Cpc%7CC8%7Cpc%7CCJ%7Cpc%7CC3%7Cpc%7CC6%7Cpc%7CDT%7Cpc%7CD2%7Cpc%7CD6%7Cpc%7CDA%7Cpc%7CHA%7Cpc%7CH3%7Cpc%7CH5%7Cpc%7CH4%7Cpc%7CSA%7Cpc%7CS8%7Cpc%7CS3%7Cpc%7CH8%7Cpc%7CD9%7Cmc%7C8%7C');this.style.color='red';return false;">Movie</A>&nbsp;or&nbsp;<A HREF="fetchlin.php?id=4442664572&when_played=1747543952">Lin</A></td>
		<td class="traveller"><A HREF="/myhands/hands.php?traveller=SB%3Aiit_bombay-1747543952-1&amp;username=iit_bombay">Traveller</A></td>
	</tr>


In [3]:
from util import parse_lin, display_lin
from urllib.parse import unquote

# lin=pn|icichacal,iit_bombay,kvbbo,AnilKhatri|st||md|1ST652HT93DAQ72CKT,S83HK7DJ983CAQJ43,SAKJ974HA84D65C62,SQHQJ652DKT4C9875|sv|e|rh||ah|Board%203|mb|P|mb|P|mb|1S|mb|P|mb|3C|mb|D|mb|4S|mb|P|mb|P|mb|P|pc|C9|pc|CT|pc|CJ|pc|C2|pc|CA|pc|C6|pc|C5|pc|CK|pc|H7|pc|H4|pc|HJ|pc|H3|pc|H2|pc|H9|pc|HK|pc|HA|pc|SA|pc|SQ|pc|S2|pc|S3|pc|SK|pc|H5|pc|S5|pc|S8|pc|D5|pc|D4|pc|DQ|pc|D3|pc|DA|pc|D8|pc|D6|pc|DT|pc|D2|pc|D9|pc|S4|pc|DK|pc|S7|pc|H6|pc|ST|pc|C3|pc|D7|pc|DJ|pc|S9|pc|C7|pc|H8|pc|HQ|pc|HT|pc|C4|pc|C8|pc|S6|pc|CQ|pc|SJ|'
# pn|icichacal,iit_bombay,kvbbo,AnilKhatri|st||md|1ST652HT93DAQ72CKT,S83HK7DJ983CAQJ43,SAKJ974HA84D65C62,SQHQJ652DKT4C9875|sv|e|rh||ah|Board%203|mb|P|mb|P|mb|1S|mb|P|mb|3C|mb|D|mb|4S|mb|P|mb|P|mb|P|pc|C9|pc|CT|pc|CJ|pc|C2|pc|CA|pc|C6|pc|C5|pc|CK|pc|H7|pc|H4|pc|HJ|pc|H3|pc|H2|pc|H9|pc|HK|pc|HA|pc|SA|pc|SQ|pc|S2|pc|S3|pc|SK|pc|H5|pc|S5|pc|S8|pc|D5|pc|D4|pc|DQ|pc|D3|pc|DA|pc|D8|pc|D6|pc|DT|pc|D2|pc|D9|pc|S4|pc|DK|pc|S7|pc|H6|pc|ST|pc|C3|pc|D7|pc|DJ|pc|S9|pc|C7|pc|H8|pc|HQ|pc|HT|pc|C4|pc|C8|pc|S6|pc|CQ|pc|SJ|
# url='https://www.bridgebase.com/tools/handviewer.html?lin=pn|icichacal,iit_bombay,kvbbo,AnilKhatri|st||md|1ST652HT93DAQ72CKT,S83HK7DJ983CAQJ43,SAKJ974HA84D65C62,SQHQJ652DKT4C9875|sv|e|rh||ah|Board%203|mb|P|mb|P|mb|1S|mb|P|mb|3C|mb|D|mb|4S|mb|P|mb|P|mb|P|pc|C9|pc|CT|pc|CJ|pc|C2|pc|CA|pc|C6|pc|C5|pc|CK|pc|H7|pc|H4|pc|HJ|pc|H3|pc|H2|pc|H9|pc|HK|pc|HA|pc|SA|pc|SQ|pc|S2|pc|S3|pc|SK|pc|H5|pc|S5|pc|S8|pc|D5|pc|D4|pc|DQ|pc|D3|pc|DA|pc|D8|pc|D6|pc|DT|pc|D2|pc|D9|pc|S4|pc|DK|pc|S7|pc|H6|pc|ST|pc|C3|pc|D7|pc|DJ|pc|S9|pc|C7|pc|H8|pc|HQ|pc|HT|pc|C4|pc|C8|pc|S6|pc|CQ|pc|SJ|'
# url='https://www.bridgebase.com/tools/handviewer.html?lin=pn|icichacal,iit_bombay,kvbbo,AnilKhatri|st||md|2S43HAQ8DT762CAKQ2,SQT976HK932DC8764,SAKJHT654DAKQ54C5,S852HJ7DJ983CJT93|sv|b|rh||ah|Board%204|mb|P|mb|1D|mb|P|mb|2D|mb|P|mb|3C|mb|P|mb|3H|mb|P|mb|4N|mb|P|mb|5H|mb|P|mb|6D|mb|P|mb|P|mb|P|pc|CJ|pc|CA|pc|C7|pc|C5|pc|D2|pc|H2|pc|DA|pc|D3|pc|DK|pc|D8|pc|D6|pc|S6|pc|DQ|pc|D9|pc|D7|pc|H3|pc|D4|pc|DJ|pc|DT|pc|S7|pc|S8|pc|S3|pc|SQ|pc|SK|pc|SA|pc|S2|pc|S4|pc|S9|pc|SJ|pc|S5|pc|C2|pc|ST|pc|H4|pc|H7|pc|HQ|pc|HK|pc|C4|mc|11|'
# url='https://www.bridgebase.com/tools/handviewer.html?lin=pn|iit_bombay,~Mwest,~Mnorth,~Meast|st||md|1SAT5HJ754DKJ3CA92,SQJ73HAQD842CK643,SK9HK9632DAQT75C8,S8642HT8D96CQJT75|sv|b|rh||ah|Board%207|mb|1C|an|Minor%20suit%20opening%20--%203+%20!C;%2011-21%20HCP;%2012-22%20total%20points|mb|P|mb|1H|an|One%20over%20one%20--%204+%20!H;%206+%20total%20points|mb|P|mb|2H|an|Simple%20raise%20--%203+%20!C;%204%20!H;%2011+%20HCP;%2012-15%20total%20points|mb|P|mb|4H|an|4+%20!H;%2012+%20HCP;%2013-18%20total%20points|mb|P|mb|P|mb|P|pc|D9|pc|DK|pc|D2|pc|D5|pc|H4|pc|HQ|pc|HK|pc|H8|pc|H2|pc|HT|pc|HJ|pc|HA|pc|D8|pc|D7|pc|D6|pc|DJ|pc|D3|pc|D4|pc|DA|pc|C5|pc|DQ|pc|S4|pc|S5|pc|S7|pc|DT|pc|C7|pc|C2|pc|C6|pc|C8|pc|CT|pc|CA|pc|C3|pc|C9|pc|CK|pc|H3|pc|CQ|pc|SK|pc|S2|pc|ST|pc|S3|pc|S9|pc|S8|pc|SA|pc|SQ|mc|12|'
url='pn%7Ciit_bombay%2C~Mwest%2C~Mnorth%2C~Meast%7Cst%7C%7Cmd%7C3SKJ9754HKTDA2CKQ3%2CS2HAQ75DKQ9854CT4%2CS83HJ8642DJ3C7652%2CSAQT6H93DT76CAJ98%7Csv%7Co%7Crh%7C%7Cah%7CBoard%201%7Cmb%7CP%7Cmb%7CP%7Cmb%7C1S%7Can%7CMajor%20suit%20opening%20--%205%2B%20!S%3B%2011-21%20HCP%3B%2012-22%20total%20points%7Cmb%7C2D%7Can%7CTwo-level%20overcall%20--%205%2B%20!D%3B%2010%2B%20HCP%3B%2011-18%20total%20points%7Cmb%7CP%7Cmb%7C2S%7Can%7CGood%20support%20in%20D%20--%203%2B%20!D%3B%2011-%20HCP%3B%2011-12%20total%20points%7Cmb%7CD%7Can%7C21-%20HCP%3B%20rebiddable%20!S%3B%2016-22%20total%20points%7Cmb%7C3D%7Can%7Ctwice%20rebiddable%20!D%3B%2012-13%20total%20points%7Cmb%7CP%7Cmb%7CP%7Cmb%7C3S%7Can%7C21-%20HCP%3B%20twice%20rebiddable%20!S%3B%2019-22%20total%20points%7Cmb%7CP%7Cmb%7CP%7Cmb%7CD%7Can%7C3%2B%20!D%3B%2011%20HCP%3B%20biddable%20!S%3B%2012-%20total%20points%7Cmb%7CP%7Cmb%7CP%7Cmb%7CP%7Cpc%7CDK%7Cpc%7CD3%7Cpc%7CD6%7Cpc%7CDA%7Cpc%7CD2%7Cpc%7CDQ%7Cpc%7CDJ%7Cpc%7CD7%7Cpc%7CHA%7Cpc%7CH2%7Cpc%7CH9%7Cpc%7CHT%7Cpc%7CH7%7Cpc%7CH8%7Cpc%7CH3%7Cpc%7CHK%7Cpc%7CCQ%7Cpc%7CC4%7Cpc%7CC2%7Cpc%7CCA%7Cpc%7CC8%7Cpc%7CCK%7Cpc%7CCT%7Cpc%7CC5%7Cpc%7CC3%7Cpc%7CS2%7Cpc%7CC6%7Cpc%7CCJ%7Cpc%7CHQ%7Cpc%7CH4%7Cpc%7CDT%7Cpc%7CS4%7Cpc%7CS5%7Cpc%7CD9%7Cpc%7CS8%7Cpc%7CST%7Cpc%7CS6%7Cpc%7CS7%7Cpc%7CD8%7Cpc%7CS3%7Cpc%7CS9%7Cpc%7CD4%7Cpc%7CC7%7Cpc%7CSQ%7Cpc%7CC9%7Cpc%7CSJ%7Cpc%7CH5%7Cpc%7CH6%7Cpc%7CSK%7Cpc%7CD5%7Cpc%7CHJ%7Cpc%7CSA%7C'
if '?' in url:
    url = url.split('?')[1]

url = unquote(url)

print(f"Encoded URL: {url}")

board = parse_lin(url)
hands = board.hands
display_lin(url)

Encoded URL: pn|iit_bombay,~Mwest,~Mnorth,~Meast|st||md|3SKJ9754HKTDA2CKQ3,S2HAQ75DKQ9854CT4,S83HJ8642DJ3C7652,SAQT6H93DT76CAJ98|sv|o|rh||ah|Board 1|mb|P|mb|P|mb|1S|an|Major suit opening -- 5+ !S; 11-21 HCP; 12-22 total points|mb|2D|an|Two-level overcall -- 5+ !D; 10+ HCP; 11-18 total points|mb|P|mb|2S|an|Good support in D -- 3+ !D; 11- HCP; 11-12 total points|mb|D|an|21- HCP; rebiddable !S; 16-22 total points|mb|3D|an|twice rebiddable !D; 12-13 total points|mb|P|mb|P|mb|3S|an|21- HCP; twice rebiddable !S; 19-22 total points|mb|P|mb|P|mb|D|an|3+ !D; 11 HCP; biddable !S; 12- total points|mb|P|mb|P|mb|P|pc|DK|pc|D3|pc|D6|pc|DA|pc|D2|pc|DQ|pc|DJ|pc|D7|pc|HA|pc|H2|pc|H9|pc|HT|pc|H7|pc|H8|pc|H3|pc|HK|pc|CQ|pc|C4|pc|C2|pc|CA|pc|C8|pc|CK|pc|CT|pc|C5|pc|C3|pc|S2|pc|C6|pc|CJ|pc|HQ|pc|H4|pc|DT|pc|S4|pc|S5|pc|D9|pc|S8|pc|ST|pc|S6|pc|S7|pc|D8|pc|S3|pc|S9|pc|D4|pc|C7|pc|SQ|pc|C9|pc|SJ|pc|H5|pc|H6|pc|SK|pc|D5|pc|HJ|pc|SA|


/Users/allwynsequeira/Sites/ben/


In [21]:

bidder_bots = [BotBid([False, False], hand, models, sampler,i, 0,False) for i, hand in enumerate(hands)]
auction = []  # since North deals, we don't need any 'PAD_START'

turn_i = 0  # whose turn is it to bid

while not bidding.auction_over(auction):
    auction.append(bidder_bots[turn_i].bid(auction).bid)
    turn_i = (turn_i + 1) % 4  # next player's turn
    
auction

['PASS', 'PASS', '1S', '2D', 'PASS', '2S', '3S', '4D', 'PASS', 'PASS', 'PASS']